# Athar wall damage model: Qwen3.5 fine-tuning on Colab

This notebook trains Athar's own damage-mapping model. It takes a rectified wall photo and answers with
strict JSON elements (`damage_type`, `geometry`, normalised `points`, `severity`, `damage_class`,
`confidence`) that the backend turns into the wall map, the treatment plan and the accuracy card.

* **Base model:** Qwen3.5 Small (Apache-2.0), natively multimodal. The 4B model is the default.
* **Method:** bf16 LoRA with Unsloth. 4-bit QLoRA is not used for Qwen3.5 because its quantisation error is
  higher than usual for this family; 4-bit appears only in the GGUF files exported for inference.
* **Where it runs:** training happens only here, on a Colab GPU (L4 22.5 GB by default, A100 when available).
  The 16 GB laptop is for inference only, through the GGUF export at the end.
* **Data:** expert-approved Athar maps (photos already face-blurred at upload), split by site so no site
  appears in more than one split, plus licence-checked public crack sets as auxiliary tasks.

Run the cells in order. The first training steps are slow while the custom Mamba Triton kernels compile.

## 1. GPU and profile

The profile is chosen from GPU memory: 40 GB or more trains the 9B model, about 22 GB trains the 4B model
(set `TRY_9B = True` to try 9B, which is at the edge of an L4), and less than 16 GB falls back to 2B.

In [ ]:
TRY_9B = False

import torch

assert torch.cuda.is_available(), "Choose Runtime > Change runtime type and pick an L4 or A100 GPU."
gpu = torch.cuda.get_device_properties(0)
VRAM_GB = gpu.total_memory / 1024**3
if VRAM_GB >= 40:
    PROFILE = "qwen35_9b_a100"
elif TRY_9B and VRAM_GB >= 20:
    PROFILE = "qwen35_9b_l4"
elif VRAM_GB >= 16:
    PROFILE = "qwen35_4b_l4"
else:
    PROFILE = "qwen35_2b_small"
print(f"{gpu.name}: {VRAM_GB:.1f} GB, bf16 support {torch.cuda.is_bf16_supported()}, profile {PROFILE}")

## 2. Drive and repository

Training outputs go to `MyDrive/athar-ml`. Put the training export there as `athar-export.zip`
(download it from Athar with `GET /api/v1/ml/training-export` using a model engineer account, or run
`python -m ml.data.export_athar` on a machine that can reach the server). Public dataset archives that need a
manual download go in `MyDrive/athar-ml/raw/<source key>/`, see `ml/data/README.md`.

For a private repository, add a Colab secret named `GITHUB_TOKEN` with read access.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE_DIR = Path("/content/drive/MyDrive/athar-ml")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path("/content/Athar")
REPO_URL = "https://github.com/Muath477/Athar.git"

try:
    token = userdata.get("GITHUB_TOKEN")
except userdata.SecretNotFoundError:
    token = None
clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", clone_url, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], check=True, capture_output=True, text=True).stdout)

## 3. Pinned libraries

Qwen3.5 needs transformers v5. The versions below are pinned in `ml/requirements-colab.txt` and match what
Unsloth 2026.11.6 accepts. Restart the runtime only if the cell asks for it.

In [ ]:
!pip install -q uv
!uv pip install --system -q -r ml/requirements-colab.txt
from importlib import metadata

for package in ("unsloth", "transformers", "trl", "peft", "torch"):
    print(package, metadata.version(package))

## 4. Dataset

Unpacks the Athar export, organises the public sets that are present, then writes chat-format
`train/val/test.jsonl` split by site. Weak labels (unedited draft maps from another detector) are added only to
training and never from a site that is held out.

In [ ]:
export_zip = DRIVE_DIR / "athar-export.zip"
assert export_zip.exists(), f"Upload the training export to {export_zip}"
!python -m ml.data.export_athar --archive {export_zip}

raw_on_drive = DRIVE_DIR / "raw"
if raw_on_drive.exists():
    shutil.copytree(raw_on_drive, "ml/data/raw", dirs_exist_ok=True)
manual_sources = [p.name for p in Path("ml/data/raw").glob("*") if p.is_dir()]
sources_arg = " ".join(sorted({"rfsat_masonry", *manual_sources}))
!python -m ml.data.collect organise --only {sources_arg}
!python -m ml.data.build_dataset --public-cap 1500
print(Path("ml/data/build/report.md").read_text())

## 5. Model and a look at one training example

Loads the profile, checks it matches the GPU choice, and prints one rendered training conversation so the
chat template used in training can be compared with the one used at inference.

In [ ]:
import json

from ml.train import train as athar_train
from ml.train.profiles import fallback_ladder, load_config, pick_profile

assert pick_profile(VRAM_GB, try_9b=TRY_9B) == PROFILE
config = load_config(PROFILE)
print(config)
print("Memory fallback ladder:")
for step in fallback_ladder(config):
    print(f"  {step.name}: image side {step.max_image_side}px, LoRA r={step.lora.r}")

sample = athar_train.read_rows(config.data.train)[0]
print(json.dumps(sample["messages"], ensure_ascii=False, indent=1)[:3000])

## 6. Training

Settings come from `ml/configs/<profile>.yaml`:

| setting | 4B on L4 (default) | 9B on L4 (edge) | 9B on A100 |
|---|---|---|---|
| precision | bf16 LoRA | bf16 LoRA | bf16 LoRA |
| LoRA r / alpha | 16 / 16 | 16 / 16 | 16 / 16 |
| vision layers | trained | frozen | trained |
| image side cap | 896 px | 672 px | 896 px |
| max sequence | 4096 | 2048 | 4096 |
| batch x accumulation | 2 x 4 | 1 x 8 | 2 x 4 |
| learning rate | 2e-4 cosine, 3% warm-up | 1e-4 cosine | 1e-4 cosine |
| epochs | 3, evaluation every epoch, best by eval loss | same | same |
| optimiser | AdamW 8-bit, weight decay 0.01, clip 1.0 | same | same |
| gradient checkpointing | `"unsloth"` | `"unsloth"` | `"unsloth"` |

Unsloth measures bf16 LoRA at about 10 GB for 4B and 22 GB for 9B before image tokens, so 9B on a 22.5 GB L4
sits at the limit. If memory runs out, training steps down on its own: smaller images first, then LoRA r=8,
then the next smaller model. An A100 is the safe choice for 9B.

Watch the training loss for the first 20 steps: it should fall quickly because the answer format is fixed.
A flat loss usually means the learning rate is too low; spikes mean it is too high.

In [ ]:
torch.cuda.reset_peak_memory_stats()
trained = athar_train.train_with_fallback(config)
config = trained.config
print(f"Trained {config.name}; peak GPU memory {torch.cuda.max_memory_reserved() / 1024**3:.1f} GB")

In [ ]:
history = [entry for entry in trained.trainer.state.log_history if "eval_loss" in entry]
for entry in history:
    print(f"epoch {entry['epoch']:.0f}: eval loss {entry['eval_loss']:.4f}")

## 7. Save the adapter and the merged 16-bit model

In [ ]:
adapter_dir = athar_train.save_adapter(trained)
merged_dir = athar_train.save_merged(trained)
run_dir = DRIVE_DIR / config.name
shutil.copytree(adapter_dir, run_dir / "adapter", dirs_exist_ok=True)
print(f"Adapter copied to {run_dir / 'adapter'}; merged model at {merged_dir}")

## 8. Evaluation on the held-out sites

Scores the model with the same cell matching as the product's accuracy card (each element is rasterised onto
the wall's grid, then `score_maps` from the backend compares cells), plus the JSON validity rate, length and
area errors, and a comparison with the detector that produced each test map's first draft.

In [ ]:
from IPython.display import Markdown, display

from ml.eval.evaluate import evaluate_rows, markdown

test_rows = [row for row in athar_train.read_rows(config.data.test) if row["task"] == "wall_map"]
predictions = athar_train.predict(trained, test_rows)
report = evaluate_rows(test_rows, predictions)
(run_dir / "metrics.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
(run_dir / "predictions.jsonl").write_text(
    "".join(json.dumps(p, ensure_ascii=False) + "\n" for p in predictions)
)
display(Markdown(markdown(report)))

## 9. GGUF for the 16 GB laptop

Converts the merged model with llama.cpp into a text GGUF and a separate `mmproj` vision projector, then
quantises to Q4_K_M (default for the laptop) and Q8_0. Both files are needed by `llama-server`. See
`ml/serve/README.md` for running them.

In [ ]:
LLAMA_CPP = Path("/content/llama.cpp")
GGUF_DIR = run_dir / "gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)
if not LLAMA_CPP.exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp {LLAMA_CPP}
!uv pip install --system -q {LLAMA_CPP}/gguf-py
!cmake -S {LLAMA_CPP} -B {LLAMA_CPP}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build {LLAMA_CPP}/build --target llama-quantize -j 8 > /dev/null

f16 = GGUF_DIR / "athar-damage-F16.gguf"
!python {LLAMA_CPP}/convert_hf_to_gguf.py {merged_dir} --outfile {f16} --outtype f16
!python {LLAMA_CPP}/convert_hf_to_gguf.py {merged_dir} --mmproj --outfile {GGUF_DIR}/mmproj-athar-damage-F16.gguf --outtype f16
for quant in ("Q4_K_M", "Q8_0"):
    !{LLAMA_CPP}/build/bin/llama-quantize {f16} {GGUF_DIR}/athar-damage-{quant}.gguf {quant}
for path in sorted(GGUF_DIR.glob("*.gguf")):
    print(f"{path.name}: {path.stat().st_size / 1024**3:.2f} GB")

## 10. Optional: push to the Hugging Face Hub

Set `PUSH_TO_HUB = True` and add a Colab secret `HF_TOKEN` with write access. Keep the repository private:
the weights were trained on site photos owned by the heritage authority.

In [ ]:
PUSH_TO_HUB = False
HUB_REPO = "your-account/athar-damage-qwen35"

if PUSH_TO_HUB:
    from huggingface_hub import HfApi

    api = HfApi(token=userdata.get("HF_TOKEN"))
    api.create_repo(HUB_REPO, private=True, exist_ok=True)
    api.upload_folder(repo_id=HUB_REPO, folder_path=str(run_dir / "adapter"), path_in_repo="adapter")
    api.upload_folder(
        repo_id=HUB_REPO,
        folder_path=str(GGUF_DIR),
        path_in_repo="gguf",
        allow_patterns=["*Q4_K_M*", "*Q8_0*", "mmproj*"],
    )

## 11. Register the version in Athar

Post this body to `POST /api/v1/ml/models` as a model engineer, then activate it from
`/dashboard/models` after comparing its metrics with the active version. The `name` must match the model
name the inference server answers to (`--alias` in llama-server, `--served-model-name` in vLLM).

In [ ]:
VERSION = "1"
registration = {
    "name": f"athar-damage-v{VERSION}",
    "version": VERSION,
    "base_model": config.base_model,
    "adapter_uri": f"hf://{HUB_REPO}/adapter" if PUSH_TO_HUB else str(run_dir / "adapter"),
    "metrics": {
        key: report[key]
        for key in ("json_valid_rate", "macro_f1", "micro_f1", "damaged_vs_sound_accuracy", "test_maps")
        if report.get(key) is not None
    },
}
print(json.dumps(registration, ensure_ascii=False, indent=2))